In [17]:
import json
import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI

In [26]:
df = pd.read_csv("data/data.csv")
documents = df.to_dict(orient="records")

In [3]:
prompt_template = """
You need to act like a user of the fitness assistant application.
Generate 5 questions that the user might ask based on the specific exercise.
Make sure the questions are specific to this exercise.
The record should contain the answer to the questions, and the questions should
be complete and not too short. Use fewer words as possible as you can from the exercise.

<exercise>

exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}

</exercise>

Provide the output in parsable JSON without using code blocks. 

Sample output:
{{"questions": ["question1", "question2", ..., "question5"]}}
""".strip()

In [15]:
client = OpenAI()

In [14]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [12]:
results = []
documents = documents[:10]

In [ ]:
for doc in tqdm(documents):
    questions = generate_questions(doc)
    parsed_questions = json.loads(questions)
    for q in parsed_questions["questions"]:
        results.append((doc["id"], q))

In [22]:
df_results = pd.DataFrame(results, columns=["doc_id", "question"])

In [23]:
df_results

,doc_id,question
0,0,What is the correct starting position for a pu...
1,0,How low should I go during a push-up?
2,0,Which muscle groups are primarily targeted by ...
3,0,Can push-ups be modified for beginners?
4,0,How can I ensure proper form while performing ...
5,1,What is the proper stance for performing squats?
6,1,How do I ensure my form is correct while lower...
7,1,Which muscle groups are primarily targeted dur...
8,1,Can squats be performed without any equipment?
9,1,What should I do if I feel discomfort while do...


In [27]:
df_results.to_csv("data/ground_truth_data.csv", index=False)